# Control policy optimization Mujoco

## Set-up

### Imports

In [1]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

Sat May 16 16:36:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.51                 Driver Version: 555.97         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   60C    P8              6W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# @title Import packages for plotting and creating graphics
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [3]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
import jax.numpy as jnp
import jax.random as jr
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

print("These device(s) are detected:", jax.devices())

These device(s) are detected: [CudaDevice(id=0)]


E0516 16:37:13.350110    1918 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0516 16:37:13.361691    1645 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


### Functions

In [4]:
from mujoco_playground import registry

class MujocoGymnaxWrapper:

    def __init__(self, env_name = None, env_instance = None, config_overrides = None):
        if env_instance is not None:
            self.env = env_instance
        elif config_overrides is not None:
            self.env = registry.load(env_name, config_overrides=config_overrides)
        else:
            self.env = registry.load(env_name)
        # self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, key, state, action, params=None):

      action = action.reshape(self.action_space)
      next_state = self.env.step(state, action)

      done = next_state.done
      obs = next_state.obs
      reward = next_state.reward

      return obs, next_state, reward, done, {}

    def render(self, rollout):
      return self.env.render(rollout)

    @property
    def dt(self):
        return self.env.dt

In [5]:
def compute_trajectory_py(key, env, policy, T=1000, stride=1):
  jit_reset = jax.jit(env.reset)
  jit_step = jax.jit(env.step)
  obs, env_state = jit_reset(key)
  states = []

  for t in range(T):
      action = policy(obs)
      obs, env_state, reward, done, _ = jit_step(key, env_state, action)

      if t % stride == 0:
          states.append(env_state)

      if done:
          break

  return states

## Cartpole

In [5]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 15

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, num_outputs_per_tree=1, device_type = 'gpu')

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4'].


In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 51.42 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -366.75347900390625, equation: [-0.0638]
Complexity: 3, fitness: -547.6666259765625, equation: [-0.0775*y3]
Complexity: 5, fitness: -978.9402465820312, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -988.626953125, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.81396484375, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -697.852783203125, equation: [0]
Complexity: 5, fitness: -978.9402465820312, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -988.626953125, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.81396484375, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -697.852783203125, equation: [0]
Complexity: 4, fitne

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 4.4*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [9]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 15

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, num_outputs_per_tree=1, device_type = 'gpu')

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4'].


In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 36.16 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -366.75347900390625, equation: [-0.0638]
Complexity: 3, fitness: -547.6666259765625, equation: [-0.0775*y3]
Complexity: 5, fitness: -978.9402465820312, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -988.626953125, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.81396484375, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -697.852783203125, equation: [0]
Complexity: 5, fitness: -978.9402465820312, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -988.626953125, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.81396484375, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -697.852783203125, equation: [0]
Complexity: 4, fitne

E0516 12:56:59.583670    6926 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit__unique_sorted_mask for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0516 12:57:59.307263    6768 slow_operation_alarm.cc:140] The operation took 2m59.782878788s

********************************
[Compiling module jit__unique_sorted_mask for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


In generation 94
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -992.9757080078125, equation: [2.17*y2]
Complexity: 5, fitness: -993.005859375, equation: [2.23*y1*y2]
Complexity: 6, fitness: -994.2534790039062, equation: [2*y2*cos(y0)]
Complexity: 7, fitness: -994.2538452148438, equation: [(y2 + sin(y2))*cos(y0)]
Complexity: 8, fitness: -994.9201049804688, equation: [y2*(y1 + cos(2*y0))]
Complexity: 9, fitness: -998.5631103515625, equation: [4.54*y2 + 4.54*y3 + 4.54*y4]
Complexity: 11, fitness: -998.585205078125, equation: [5.05*y2 + 2.02*y3 + 2.02*y4]
Complexity: 13, fitness: -999.798828125, equation: [y0 + 4.52*y2 + y3 + 1.51*y4]
Complexity: 15, fitness: -999.8280639648438, equation: [y0 + 6.03*y2 + y3 + 1.51*y4]
invalid solutions: 0 0
In generation 95
Complexity: 1, fitness: -477.8402404785156, equation: [y2]
Complexity: 3, fitness: -992.9757080078125, equation: [2.17*y2]
Complexity: 5, fitness: -993.005859375, equation: [2.23*y1*y2]
Complexity: 6

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.75*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Reacher

In [6]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy', config_overrides={'njmax': 500, 'naconmax': 500})

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


### 25 generations

In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1.91 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 7.83 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 1.93 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 1.60 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 1.32 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8f5355c load on device 'cuda:0' took 1.80 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 1.15 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 64.54 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.solver b116efb load on 

In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 64.78 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -49.5, equation: [-0.0638, 0]
Complexity: 2, fitness: -62.5, equation: [sin(y4), 0]
Complexity: 3, fitness: -98.0, equation: [y0*y3, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 6, fitness: -346.9375, equation: [y3*cos(2*y2), 2*y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -149.0625, equation: [y3, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 5, fitness: -239.125, equation: [y0*y2, y0*y2 + y3]
Complexity: 6, fitness: -346.9375, equation: [y3*cos(2*y2), 2*y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -176.1875, equation: [y2, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 5, fit

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([2*obs[3]**2 + 5*obs[3], -2.26*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3]*obs[5] - 2.26*obs[3]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [9]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5'].


In [10]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 38.76 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -49.5, equation: [-0.0638, 0]
Complexity: 2, fitness: -62.5, equation: [sin(y4), 0]
Complexity: 3, fitness: -98.0, equation: [y0*y3, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 6, fitness: -346.9375, equation: [y3*cos(2*y2), 2*y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -149.0625, equation: [y3, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 5, fitness: -239.125, equation: [y0*y2, y0*y2 + y3]
Complexity: 6, fitness: -346.9375, equation: [y3*cos(2*y2), 2*y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -176.1875, equation: [y2, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -197.0, equation: [y2 + y3 + sin(y2 + y3), 0]
Complexity: 5, fit

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([obs[3]*obs[5] + 6*obs[3], -4.53*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3] - 0.245])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Swimmer

In [11]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)

In [12]:
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)

### 25 generations

In [13]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f9126636 4908da2 load on device 'cuda:0' took 69.22 ms  (cached)
Module solve_init_jaref__locals__kernel_6e8468b3 6e8468b load on device 'cuda:0' took 2.28 ms  (cached)
Module mul_m_dense__locals___mul_m_dense_f619f9d8 f619f9d load on device 'cuda:0' took 2.78 ms  (cached)
Module update_constraint_gauss_cost__locals__kernel_84cf3b0a 84cf3b0 load on device 'cuda:0' took 2.29 ms  (cached)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_dc766f91 aa19e9d load on device 'cuda:0' took 5.35 ms  (cached)
Module update_gradient_cholesky__locals__kernel_8204ed89 a39c2f0 load on device 'cuda:0' took 101.09 ms  (cached)
Module linesearch_iterative__locals__kernel_94acdc55 9c914fd load on device 'cuda:0' took 3.83 ms  (cached)
Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12'].


In [14]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 55.47 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -159.62271118164062, equation: [0, 0.639]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 7, fitness: -172.97943115234375, equation: [y2 + y3 + y9 - 0.341, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 4, fitness: -167.80514526367188, equation: [sin(y2), y8 - sin(y2)]
Complexity: 7, fitness: -172.97943115234375, equation: [y2 + y3 + y9 - 0.341, 0]
Complexity: 10, fitness: -174.34634399414062, equation: [y3*y9, -y2 + y3*y9 + y6 + sin(y7)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 3, fitness: -171.60281372070312, equation: [2*y2, 0]
Complexity: 5, fitness: -174.15911865234375, equation: [2*y0*y3 + y2, 0]
Complexity: 10, fitness: -174.34634399414062, equation: [y3*y9, -y2 

### Simulation

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim
stride = 2

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-obs[11] - obs[2] + jnp.sin(jnp.sin(2*obs[9])), obs[1]*obs[3] + obs[2] + (2*obs[0] + obs[2])*(-obs[3] + obs[4])])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [15]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)

In [16]:
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)

In [17]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12'].


In [18]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 52.58 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -159.62271118164062, equation: [0, 0.639]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 7, fitness: -172.97943115234375, equation: [y2 + y3 + y9 - 0.341, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 4, fitness: -167.80514526367188, equation: [sin(y2), y8 - sin(y2)]
Complexity: 7, fitness: -172.97943115234375, equation: [y2 + y3 + y9 - 0.341, 0]
Complexity: 10, fitness: -174.34634399414062, equation: [y3*y9, -y2 + y3*y9 + y6 + sin(y7)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 3, fitness: -171.60281372070312, equation: [2*y2, 0]
Complexity: 5, fitness: -174.15911865234375, equation: [2*y0*y3 + y2, 0]
Complexity: 10, fitness: -174.34634399414062, equation: [y3*y9, -y2 

### Simulation

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim
stride = 2
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7] + obs[9] + jnp.sin(obs[2] - obs[6]), obs[0]*obs[2] + 2*obs[0] + 2*obs[1]*obs[3]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Hopper

In [19]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')

### 25 generations

In [20]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=4)

Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 13.33 ms  (cached)
Module _nxn_broadphase__locals__kernel_c763f4cb c763f4c load on device 'cuda:0' took 3.99 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_052aeb65 052aeb6 load on device 'cuda:0' took 3.60 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f956cf88 226c0f7 load on device 'cuda:0' took 113.71 ms  (cached)
Module solve_init_jaref__locals__kernel_173cbb76 173cbb7 load on device 'cuda:0' took 5.71 ms  (cached)
Module mul_m_dense__locals___mul_m_dense_d21c2e6d d21c2e6 load on device 'cuda:0' took 5.15 ms  (cached)
Module update_constraint_gauss_cost__locals__kernel_b1449efe b1449ef load on device 'cuda:0' took 2.86 ms  (cached)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_83d2c797 52c34bc load on device 'cuda:0' took 4.57 ms  (cached)
Module update_gradient_cholesky__locals__kernel_6f68b037 18ece50 loa

In [21]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 88.69 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -0.004872772842645645, equation: [0, 0.639, 0, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.73494553565979, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -0.757794976234436, equation: [0, y10 + y5 + y8, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -0.03512163832783699, equation: [0, 0, -1.55, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.73494553565979, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -0.757794976234436, equation: [0, y10 + y5 + y8, 0, 0]
Complexity: 7, fitness: -1.0159389972686768, equation: [0, y10*(y14 - y9 + 1.03), y9 - 1.03, 0]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -0.08202669024467468, equation: [0, 0, y8, 0]
Complexity: 3, fitness: -0.7349

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[4], 2*obs[0] + obs[8]*(obs[8] - obs[9]), 2*obs[1] - obs[2] + obs[5], obs[1] + 2*obs[8]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [22]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=4)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14'].


In [23]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 55.01 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -0.004872772842645645, equation: [0, 0.639, 0, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.5936034917831421, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -0.8270514011383057, equation: [0, y10 + y5 + y8, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -0.03512163832783699, equation: [0, 0, -1.55, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.5936034917831421, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -0.8270514011383057, equation: [0, y10 + y5 + y8, 0, 0]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -0.08202669024467468, equation: [0, 0, y8, 0]
Complexity: 3, fitness: -0.5936034917831421, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -0.8270514011383057, e

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[2]*(-2.43*obs[2] - 1.43), obs[10]*(obs[10] - 2*obs[12] + 2*obs[4]) + obs[10] - obs[6], 2*obs[1] - obs[2] - obs[6]**2 - obs[6] + obs[8] + 0.626, 2*obs[1] + obs[13] + 3*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Walker

In [6]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 2.63 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 11.84 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 2.61 ms  (cached)
Module _nxn_broadphase__locals__kernel_c763f4cb c763f4c load on device 'cuda:0' took 2.65 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_0b4c62fd 0b4c62f load on device 'cuda:0' took 976.31 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 3.94 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 3.40 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 2.99 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 18a78e5 load on devic

In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 112.66 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -24.696701049804688, equation: [0, 0.839, 0, 0, 0, 0]
Complexity: 3, fitness: -42.06257629394531, equation: [0, y19 + y22, 0, 0, 0, 0]
Complexity: 5, fitness: -59.26777648925781, equation: [y18 + y3 + y4, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -77.9920883178711, equation: [0, y0*y5*y9, 0, y0*y5*y9 + y17, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -42.90726089477539, equation: [0, 0, 0, y18, 0, 0]
Complexity: 3, fitness: -60.98970413208008, equation: [0, 0, 0, y12 + y17, 0, 0]
Complexity: 6, fitness: -66.64216613769531, equation: [y18 + 2*y22, 2*y22, 0, 0, sin(y18 + 2*y22), 0]
Complexity: 7, fitness: -77.9920883178711, equation: [0, y0*y5*y9, 0, y0*y5*y9 + y17, 0, 0]
Complexity: 8, fitness: -133.56637573242188, equation: [y14 + y21 - 0.854*y22, 0, 0, -0.854*y22, cos(y14 + y21 - 0.854*y22), 0]
invalid solutions: 0 0

### Simulation

In [ ]:
stride = 3
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([obs[15] + 2*obs[18] + obs[21], obs[9]*(-0.799*obs[19] - 0.799*obs[21]), obs[1] + obs[15], 2*obs[18] + jnp.cos(obs[14]), obs[1]*obs[10]*obs[17]**2, obs[11] + obs[14] + obs[5]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [9]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16', 'y17', 'y18', 'y19', 'y20', 'y21', 'y22', 'y23'].


In [10]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 89.62 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -24.65665054321289, equation: [0, 0.839, 0, 0, 0, 0]
Complexity: 2, fitness: -24.65868377685547, equation: [0, 0, 0, 0, sin(y19), 0]
Complexity: 3, fitness: -40.49315643310547, equation: [0, y19 + y22, 0, 0, 0, 0]
Complexity: 5, fitness: -57.13121032714844, equation: [y18 + y3 + y4, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -70.82460021972656, equation: [0, y0*y5*y9, 0, y0*y5*y9 + y17, 0, 0]
Complexity: 11, fitness: -92.39718627929688, equation: [y16 + y6 + (y15 + y23)*(-y3 + y6), 0, y15 + y23, (y15 + y23)*(-y3 + y6), -y3 + y6, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -39.19313430786133, equation: [0, 0, 0, y18, 0, 0]
Complexity: 3, fitness: -60.987159729003906, equation: [0, 0, 0, y12 + y17, 0, 0]
Complexity: 6, fitness: -63.99126052856445, equation: [y18 + 2*y22, 2*y22, 0, 0, sin(y18 + 2*y22), 0]
Complexity: 7, fitne

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([3*obs[18] - 0.455, obs[10] - obs[13] - obs[19] + obs[22], 2*obs[17], obs[18], obs[0] + obs[14] - 2*obs[8], 2*obs[20]**2])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Cheetah

In [13]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')

In [14]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16'].


In [15]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 38.48 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -0.5508458614349365, equation: [0, 0.839, 0, 0, 0, 0]
Complexity: 2, fitness: -15.274834632873535, equation: [0, 0, 0, 0, sin(y13), 0]
Complexity: 3, fitness: -76.20889282226562, equation: [y6 + y8, 0, 0, 0, 0, 0]
Complexity: 8, fitness: -101.85005950927734, equation: [y11 + y15 + cos(y3), 0, 0, y11 + y15, 0, y1 + y11 + y15 + cos(y3)]
Complexity: 9, fitness: -150.92227172851562, equation: [y0 + y11 + y14*y7 + y16, 0, 0, 0, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -129.1995391845703, equation: [y15, 0, 0, 0, 0, 0]
Complexity: 8, fitness: -132.0604248046875, equation: [0, 0, -y11 + y13 - y2 + sin(y1), 0, -y11 + y13 - y2, 0]
Complexity: 9, fitness: -158.2477264404297, equation: [y1*(y11 + y16) + y11 + y4, 0, y11 + y16, y1*(y11 + y16), 0, 0]
Complexity: 11, fitness: -172.5955352783203, equation: [y11 + y4, 0, y11 + y

### Simulation

In [ ]:
stride = 3
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([(obs[11] + jnp.cos(obs[11]))*jnp.cos(obs[5]), obs[2]**3*obs[5], obs[11]*obs[2] + obs[11] - obs[2], 0, -obs[1] + obs[3]*obs[8] + obs[3], obs[10] + jnp.cos(jnp.cos(jnp.cos(obs[6])))])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [16]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16'].


In [17]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 37.77 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -0.5508455038070679, equation: [0, 0.839, 0, 0, 0, 0]
Complexity: 2, fitness: -15.30801773071289, equation: [0, 0, 0, 0, sin(y13), 0]
Complexity: 3, fitness: -76.18108367919922, equation: [y6 + y8, 0, 0, 0, 0, 0]
Complexity: 8, fitness: -104.44615936279297, equation: [y11 + y15 + cos(y3), 0, 0, y11 + y15, 0, y1 + y11 + y15 + cos(y3)]
Complexity: 9, fitness: -154.56898498535156, equation: [y0 + y11 + y14*y7 + y16, 0, 0, 0, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -124.98069763183594, equation: [y15, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -127.96458435058594, equation: [y1*(y11 + y4) + y16, 0, 0, 0, 0, 0]
Complexity: 8, fitness: -137.8824920654297, equation: [0, 0, -y11 + y13 - y2 + sin(y1), 0, -y11 + y13 - y2, 0]
Complexity: 9, fitness: -154.56898498535156, equation: [y0 + y11 + y14*y7 + y16, 0, 0, 0, 0, 0]
Comple

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([-obs[0] - 2*obs[12] + obs[16], jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[4])))))), obs[10] + 3*obs[11], 0.334*obs[4] + 0.112, obs[12], jnp.cos(jnp.cos(3*obs[7]))])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Multiple evaluations

In [ ]:
T = 1000

def repeated_evaluation(key, policy, env):
    def single_run(key):
        obs, env_state = env.reset(key)
        (_, _, _, _), (_, treward, _, _) = jax.lax.scan(
            step_fn(policy, env),
            (key, obs, env_state, False),
            jnp.arange(T)
        )
        return jnp.sum(treward)

    keys = jr.split(key, 10)
    return jax.vmap(single_run)(keys)

def step_fn(policy, env):

    def _step(carry, t):
        key, obs, env_state, done = carry
        action = policy(obs)
        key, subkey = jr.split(key)
        obs, env_state, reward, done_new, _ = env.step(
            subkey,
            env_state,
            action,
            None
        )

        return (key, obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

### CartPole

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 4.4*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.75*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Reacher

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([2*obs[3]**2 + 5*obs[3], -2.26*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3]*obs[5] - 2.26*obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([obs[3]*obs[5] + 6*obs[3], -4.53*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3] - 0.245])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Swimmer

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-obs[11] - obs[2] + jnp.sin(jnp.sin(2*obs[9])), obs[1]*obs[3] + obs[2] + (2*obs[0] + obs[2])*(-obs[3] + obs[4])])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7] + obs[9] + jnp.sin(obs[2] - obs[6]), obs[0]*obs[2] + 2*obs[0] + 2*obs[1]*obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Hopper

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[4], 2*obs[0] + obs[8]*(obs[8] - obs[9]), 2*obs[1] - obs[2] + obs[5], obs[1] + 2*obs[8]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[2]*(-2.43*obs[2] - 1.43), obs[10]*(obs[10] - 2*obs[12] + 2*obs[4]) + obs[10] - obs[6], 2*obs[1] - obs[2] - obs[6]**2 - obs[6] + obs[8] + 0.626, 2*obs[1] + obs[13] + 3*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Walker

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([obs[15] + 2*obs[18] + obs[21], obs[9]*(-0.799*obs[19] - 0.799*obs[21]), obs[1] + obs[15], 2*obs[18] + jnp.cos(obs[14]), obs[1]*obs[10]*obs[17]**2, obs[11] + obs[14] + obs[5]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([3*obs[18] - 0.455, obs[10] - obs[13] - obs[19] + obs[22], 2*obs[17], obs[18], obs[0] + obs[14] - 2*obs[8], 2*obs[20]**2])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Cheetah

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([(obs[11] + jnp.cos(obs[11]))*jnp.cos(obs[5]), obs[2]**3*obs[5], obs[11]*obs[2] + obs[11] - obs[2], 0, -obs[1] + obs[3]*obs[8] + obs[3], obs[10] + jnp.cos(jnp.cos(jnp.cos(obs[6])))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([-obs[0] - 2*obs[12] + obs[16], jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[4])))))), obs[10] + 3*obs[11], 0.334*obs[4] + 0.112, obs[12], jnp.cos(jnp.cos(3*obs[7]))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)